In [1]:
import pandas as pd

df_rank = pd.read_csv('../catboost-classifier/datasets/rank_dataset_TRAIN_CATBOOST.csv')
df_mapping = pd.read_csv('./exercises_base_FINAL_CLEANED.csv')
df_profiles = pd.read_csv('./powerlifting_profiles_clean.csv')

df_stage_1 = df_rank.merge(
    df_mapping[['exercise_id', 'Base_Lift', 'Exercise_Name']], 
    left_on='candidate_id', 
    right_on='exercise_id', 
    how='inner' # Оставляем только то, что разметили и что есть в трейне
)

final_master_df = df_stage_1.join(df_profiles, how='inner')

final_master_df.to_csv('MASTER_CATBOOST_DATASET.csv', index=False)

print(f"Сборка завершена!")
print(f"Финальное количество строк: {len(final_master_df)}")
display(final_master_df.head())

Сборка завершена!
Финальное количество строк: 45239


,bert_score,candidate_id,is_same_group,target,eq_clean,level_idx,goal_clean,exercise_id,Base_Lift,Exercise_Name,Sex,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg
0,0.001278,1546,0,0,Machine,3,Powerbuilding,1546,Best3BenchKg,Low Cable Fly,F,33.0,58.30,80.0,60.0,107.5
1,0.001276,1440,0,0,Machine,0,Bodybuilding,1440,Best3DeadliftKg,Lat Pullover,F,43.0,73.10,105.0,67.5,110.0
2,0.001778,2733,0,0,Machine,0,Athletics,2733,Best3DeadliftKg,Super ROM Overhead Cable Row,M,15.5,67.40,100.0,62.5,105.0
3,0.001300,2731,0,0,Dumbbell,0,Athletics,2731,Best3DeadliftKg,Super ROM Lateral Dumbbell Raise,M,35.0,66.65,137.5,122.5,170.0
4,0.001246,1075,0,0,Barbell,1,Athletics,1075,Best3BenchKg,Half Kneeling Landmine Press,M,26.5,72.45,90.0,50.0,125.0


In [14]:
final_master_df['eq_clean'].unique()

array(['Machine', 'Dumbbell', 'Barbell', 'Bodyweight'], dtype=object)

In [2]:
features_with_id = [
    'exercise_id',
    'Sex', 'Age', 'BodyweightKg', 
    'eq_clean', 'level_idx', 'goal_clean', 
    'Base_Lift', 
    'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg'
]

df_final_reg = final_master_df[features_with_id].copy()

df_final_reg['exercise_id'] = df_final_reg['exercise_id'].astype(int)

print(f"Теперь всё на месте. Размер: {df_final_reg.shape}")
display(df_final_reg.head(20))

Теперь всё на месте. Размер: (45239, 11)


,exercise_id,Sex,Age,BodyweightKg,eq_clean,level_idx,goal_clean,Base_Lift,Best3SquatKg,Best3BenchKg,Best3DeadliftKg
0,1546,F,33.0,58.30,Machine,3,Powerbuilding,Best3BenchKg,80.0,60.0,107.5
1,1440,F,43.0,73.10,Machine,0,Bodybuilding,Best3DeadliftKg,105.0,67.5,110.0
2,2733,M,15.5,67.40,Machine,0,Athletics,Best3DeadliftKg,100.0,62.5,105.0
3,2731,M,35.0,66.65,Dumbbell,0,Athletics,Best3DeadliftKg,137.5,122.5,170.0
4,1075,M,26.5,72.45,Barbell,1,Athletics,Best3BenchKg,90.0,50.0,125.0
5,1134,M,15.5,78.80,Dumbbell,2,Powerbuilding,Best3DeadliftKg,100.0,60.0,115.0
6,2577,M,57.5,79.65,Dumbbell,2,Powerlifting,Best3DeadliftKg,180.0,100.0,55.0
7,185,F,26.0,96.50,Bodyweight,2,Muscle & Sculpting,Cardio_Core,100.0,47.5,140.0
8,2703,M,31.5,102.55,Barbell,2,Powerlifting,Best3DeadliftKg,232.5,160.0,260.0
9,1472,F,35.0,53.60,Dumbbell,0,Muscle & Sculpting,Best3DeadliftKg,80.0,50.0,92.5


In [3]:
# Нам нужно вытащить Exercise_Name из твоего маппинга по exercise_id
# Используем merge, чтобы точно сопоставить названия с ID
df_final_reg = df_final_reg.merge(
    df_mapping[['exercise_id', 'Exercise_Name']], 
    on='exercise_id', 
    how='left'
)

# Переставим колонки, чтобы название было сразу после ID для удобства
cols = ['exercise_id', 'Exercise_Name'] + [c for c in df_final_reg.columns if c not in ['exercise_id', 'Exercise_Name']]
df_final_reg = df_final_reg[cols]

print(f"Теперь таблица 'говорящая'. Размер: {df_final_reg.shape}")
display(df_final_reg.head())

Теперь таблица 'говорящая'. Размер: (45239, 12)


,exercise_id,Exercise_Name,Sex,Age,BodyweightKg,eq_clean,level_idx,goal_clean,Base_Lift,Best3SquatKg,Best3BenchKg,Best3DeadliftKg
0,1546,Low Cable Fly,F,33.0,58.30,Machine,3,Powerbuilding,Best3BenchKg,80.0,60.0,107.5
1,1440,Lat Pullover,F,43.0,73.10,Machine,0,Bodybuilding,Best3DeadliftKg,105.0,67.5,110.0
2,2733,Super ROM Overhead Cable Row,M,15.5,67.40,Machine,0,Athletics,Best3DeadliftKg,100.0,62.5,105.0
3,2731,Super ROM Lateral Dumbbell Raise,M,35.0,66.65,Dumbbell,0,Athletics,Best3DeadliftKg,137.5,122.5,170.0
4,1075,Half Kneeling Landmine Press,M,26.5,72.45,Barbell,1,Athletics,Best3BenchKg,90.0,50.0,125.0


In [15]:
# Список столбцов, которые критически важны для логики весов
important_categorical_cols = ['Sex', 'goal_clean', 'level_idx', 'eq_clean', 'Base_Lift']

print("🔍 Проверка уникальных значений в категориальных столбцах:\n")

for col in important_categorical_cols:
    if col in df_final_reg.columns:
        unique_vals = df_final_reg[col].unique()
        print(f"🔹 Столбец: {col}")
        print(f"   Значения ({len(unique_vals)}): {unique_vals}")
        print("-" * 30)
    else:
        print(f"❌ Столбец '{col}' не найден в датасете!")

# Также проверим наличие ключевых слов в названиях упражнений для иерархии
print("\n📝 Краткий пример названий упражнений (Exercise_Name):")
print(df_final_reg['Exercise_Name'].head(10).tolist())

🔍 Проверка уникальных значений в категориальных столбцах:

🔹 Столбец: Sex
   Значения (3): ['F' 'M' 'Mx']
------------------------------
🔹 Столбец: goal_clean
   Значения (7): ['Powerbuilding' 'Bodybuilding' 'Athletics' 'Powerlifting'
 'Muscle & Sculpting' 'Bodyweight Fitness' 'Fitness']
------------------------------
🔹 Столбец: level_idx
   Значения (4): [3 0 1 2]
------------------------------
🔹 Столбец: eq_clean
   Значения (4): ['Machine' 'Dumbbell' 'Barbell' 'Bodyweight']
------------------------------
🔹 Столбец: Base_Lift
   Значения (4): ['Best3BenchKg' 'Best3DeadliftKg' 'Cardio_Core' 'Best3SquatKg']
------------------------------

📝 Краткий пример названий упражнений (Exercise_Name):
['Low Cable Fly', 'Lat Pullover', 'Super ROM Overhead Cable Row', 'Super ROM Lateral Dumbbell Raise', 'Half Kneeling Landmine Press', 'Heavy Lateral Raise', 'Snatch (Dumbbell)', 'B-Skips', 'Straight Leg Deadlift', 'Lateral Raise (Seated)']


In [10]:
# import pandas as pd
# import numpy as np

# def calculate_pro_stats_integrated(row):
#     # --- 1. ПАРАМЕТРЫ ЦЕЛИ ---
#     goal_logic = {
#         'Powerlifting': {'reps': 3, 'int': 0.85, 'scale': 1.1},
#         'Bodybuilding': {'reps': 10, 'int': 0.70, 'scale': 1.0},
#         'Athletics': {'reps': 12, 'int': 0.62, 'scale': 0.85},
#         'Powerbuilding': {'reps': 8, 'int': 0.75, 'scale': 1.05},
#         'Muscle & Sculpting': {'reps': 15, 'int': 0.55, 'scale': 0.9},
#         'Fitness': {'reps': 12, 'int': 0.60, 'scale': 0.8}
#     }
#     g_params = goal_logic.get(row['goal_clean'], {'reps': 10, 'int': 0.70, 'scale': 0.9})
    
#     # --- 2. СИСТЕМНЫЕ ИСКЛЮЧЕНИЯ (ВЕС = 0) ---
#     ex_name_lower = str(row['Exercise_Name']).lower()
#     no_weight_list = ['plank', 'run', 'skip', 'jump', 'stretch', 'yoga', 'cardio', 'pogo', 'walking']
#     if any(kw in ex_name_lower for kw in no_weight_list):
#         return 0.0, int(g_params['reps'])

#     # --- 3. БИОЛОГИЧЕСКИЙ РАСЧЕТ (LIGHT vs PRO) ---
#     level_idx = int(row['level_idx'])
#     is_light = (row.get('Best3SquatKg', 0) + row.get('Best3BenchKg', 0) + row.get('Best3DeadliftKg', 0)) == 0
    
#     if is_light:
#         # ПРАВИЛО ЭФФЕКТИВНОЙ МАССЫ
#         eff_mass_ratio = 0.25 + (level_idx * 0.20)
#         base_capacity = row['BodyweightKg'] * eff_mass_ratio
#     else:
#         # PRO режим
#         base_capacity = row[row['Base_Lift']] if row['Base_Lift'] in row else 0

#     # --- 4. ГЕНДЕРНЫЙ ВЕКТОР ---
#     gender_mult = 1.0
#     if row['Sex'] == 'F':
#         is_upper = 'bench' in str(row['Base_Lift']).lower()
#         gender_mult = 0.55 if is_upper else 0.85 

#     # --- 5. ИЕРАРХИЯ СЛОЖНОСТИ (V-Factor) ---
#     base_keyword = str(row['Base_Lift']).replace('Best3', '').replace('Kg', '').lower()
#     is_direct_base = base_keyword in ex_name_lower and 'barbell' in str(row['eq_clean']).lower()
    
#     eq_type = str(row['eq_clean']).lower()
#     if is_direct_base:
#         v_factor = 1.05
#     elif 'barbell' in eq_type:
#         v_factor = 0.85
#     elif 'machine' in eq_type:
#         v_factor = 1.15
#     elif 'dumbbell' in eq_type:
#         v_factor = 0.75
#     else:
#         v_factor = 0.8

#     # ШТРАФ ЗА ИЗОЛЯЦИЮ
#     compound_patterns = ['press', 'squat', 'deadlift', 'row', 'lunge', 'clean', 'snatch', 'thrust']
#     if not any(p in ex_name_lower for p in compound_patterns):
#         v_factor *= 0.25

#     # --- 6. РАСЧЕТ И КОРРЕКЦИЯ ПО ОБОРУДОВАНИЮ (НОВОЕ) ---
#     weight = base_capacity * gender_mult * v_factor * g_params['int'] * g_params['scale']
    
#     # Поправка на возраст
#     age_adj = 1.0 - (max(0, row['Age'] - 40) * 0.005)
#     weight *= age_adj

#     # ПРАВИЛО МИНИМАЛЬНОГО ВЕСА ПО ТИПУ СНАРЯДА
#     if 'machine' in eq_type:
#         # В тренажере минимум 5 кг, если расчет дал меньше
#         weight = max(5.0, weight)
#     elif 'dumbbell' in eq_type:
#         # Гантели минимум 2 кг, шаг 2 кг
#         weight = max(2.0, round(weight / 2) * 2)
#     elif 'barbell' in eq_type:
#         # Штанга: если это база - минимум 20 кг, если подсобка - можно меньше
#         min_bar = 20.0 if is_direct_base else 5.0
#         weight = max(min_bar, weight)

#     # --- 7. КОРРЕКЦИЯ ДЛЯ BODYWEIGHT ---
#     if eq_type == 'bodyweight':
#         if any(x in ex_name_lower for x in ['pull', 'chin', 'dip', 'push up']):
#             # Здесь оставляем минус, так как это помощь (Гравитрон)
#             weight = weight - (row['BodyweightKg'] * 0.85)
#             weight = round(weight, 1)
#         else:
#             # Пресс/Косые - убираем микро-веса типа 0.3 кг
#             weight = 0.0 if (level_idx < 2 or weight < 2.0) else round(weight, 1)
#     else:
#         weight = round(max(0, weight), 1)

#     return weight, int(g_params['reps'] + (2 if row['Sex'] == 'F' else 0))

In [11]:
import os

# Применяем расчет ко ВСЕМУ датасету
print(f"🔄 Обработка {len(df_final_reg)} строк. Пожалуйста, подождите...")

# Применяем функцию и разделяем результат на две колонки
results_all = df_final_reg.apply(lambda x: calculate_pro_stats_integrated(x), axis=1)
df_final_reg['predicted_weight'] = [x[0] for x in results_all]
df_final_reg['target_reps'] = [x[1] for x in results_all]

# Формируем имя файла
output_filename = "./datasets/gym_dataset_with_targets_FINAL.csv"

# Сохраняем на диск
df_final_reg.to_csv(output_filename, index=False)

print(f"✅ Готово! Датасет сохранен как: {output_filename}")
print(f"📊 Средний предсказанный вес: {df_final_reg['predicted_weight'].mean():.2f} кг")

🔄 Обработка 45239 строк. Пожалуйста, подождите...
✅ Готово! Датасет сохранен как: ./datasets/gym_dataset_with_targets_FINAL.csv
📊 Средний предсказанный вес: 39.79 кг


In [19]:
import pandas as pd
import numpy as np

def calculate_final_logic(row):
    # --- 1. ЦЕЛИ (Повторы и Интенсивность по твоему анализу) ---
    goal_map = {
        'Powerlifting':      {'reps': 3,  'int': 0.88}, # Макс. сила
        'Powerbuilding':     {'reps': 8,  'int': 0.78}, # Сила + Объем
        'Bodybuilding':      {'reps': 10, 'int': 0.72}, # Гипертрофия
        'Athletics':         {'reps': 12, 'int': 0.60}, # Выносливость
        'Muscle & Sculpting':{'reps': 14, 'int': 0.55}, # Рельеф
        'Fitness':           {'reps': 12, 'int': 0.60}, # ОФП
        'Bodyweight Fitness':{'reps': 12, 'int': 0.60}  # Работа с весом тела
    }
    params = goal_map.get(row['goal_clean'], {'reps': 10, 'int': 0.70})
    
    # --- 2. ГЕНДЕРНЫЙ ОБЪЕМ (Повторы) ---
    # По статистике: F (+2), Mx (+1), M (0)
    extra_reps = {'F': 2, 'Mx': 1, 'M': 0}.get(row['Sex'], 0)
    final_reps = params['reps'] + extra_reps

    # --- 3. ЭФФЕКТИВНОСТЬ МАССЫ (Вес тела + Уровень) ---
    # Реализация твоей идеи: Новичок (22% КПД) -> Профи (88% КПД)
    level_eff = {0: 0.22, 1: 0.42, 2: 0.65, 3: 0.88}
    level_factor = level_eff.get(row['level_idx'], 0.45)
    
    # Проверка режима (PRO или LIGHT)
    sbd_sum = sum([row.get(c, 0) for c in ['Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg'] if pd.notna(row.get(c))])
    is_pro = sbd_sum > 0
    
    if is_pro and row['Base_Lift'] != 'Cardio_Core':
        base_power = row[row['Base_Lift']] if row['Base_Lift'] in row else row['BodyweightKg'] * level_factor
    else:
        # Для новичков без рекордов: "Силовая база" = Вес тела * КПД уровня
        base_power = row['BodyweightKg'] * level_factor

    # --- 4. ГЕНДЕРНЫЙ СИЛОВОЙ ВЕКТОР (Физиология) ---
    gender_vector = 1.0
    if row['Sex'] in ['F', 'Mx']:
        # Женский верх тела пропорционально слабее низа
        is_upper = 'bench' in str(row['Base_Lift']).lower()
        if is_upper:
            gender_vector = 0.55 if row['Sex'] == 'F' else 0.75
        else:
            gender_vector = 0.85 if row['Sex'] == 'F' else 0.92

    # --- 5. ОБОРУДОВАНИЕ И ИЕРАРХИЯ СЛОЖНОСТИ ---
    eq = str(row['eq_clean']).lower()
    ex = str(row['Exercise_Name']).lower()
    
    # Базовые коэффициенты снарядов
    eq_mult = {'barbell': 1.0, 'dumbbell': 0.75, 'machine': 1.15, 'bodyweight': 1.0}.get(eq, 0.8)

    # Авто-определение изоляции (штраф к весу без привязки к названиям)
    compounds = ['press', 'squat', 'deadlift', 'row', 'lunge', 'clean', 'snatch', 'thrust', 'bench']
    if not any(word in ex for word in compounds):
        eq_mult *= 0.28 # Бицепс, махи и т.д. получают 28% от базы

    # --- 6. ИТОГОВЫЙ РАСЧЕТ ВЕСА ---
    weight = base_power * gender_vector * eq_mult * params['int']
    
    # Возрастная коррекция (после 40 лет)
    if row['Age'] > 40:
        weight *= (1.0 - (row['Age'] - 40) * 0.005)

    # --- 7. ПРАВИЛА ОБОРУДОВАНИЯ (Твои "Законы зала") ---
    if eq == 'dumbbell':
        # Вес ОДНОЙ гантели, шаг 2 кг, минимум 2 кг
        weight = max(2.0, round((weight / 2) / 2) * 2)
    elif eq == 'machine':
        # Минимум 5 кг, шаг 5 кг
        weight = max(5.0, round(weight / 5) * 5)
    elif eq == 'barbell':
        # База минимум 20кг (гриф), подсобка - от 5кг
        is_base_ex = any(b in ex for b in ['bench', 'squat', 'deadlift'])
        weight = max(20.0 if is_base_ex else 5.0, round(weight / 2.5) * 2.5)
    elif eq == 'bodyweight':
        # Помощь в подтягиваниях (минус) или доп. вес для профи
        if any(x in ex for x in ['pull', 'chin', 'dip', 'push up']):
            weight = weight - (row['BodyweightKg'] * 0.85)
        else:
            weight = 0.0 if row['level_idx'] < 2 else weight * 0.15

    return round(weight, 1), int(final_reps)

# ЗАПУСК ТРАНСФОРМАЦИИ
print("🔩 Запускаем процесс генерации weight_target и reps_target...")
final_results = df_final_reg.apply(calculate_final_logic, axis=1)

df_final_reg['weight_target'] = [x[0] for x in final_results]
df_final_reg['reps_target'] = [x[1] for x in final_results]

print("✅ Столбцы успешно добавлены.")
print("\n📊 Контрольный срез данных:")
display(df_final_reg.sample(15))

🔩 Запускаем процесс генерации weight_target и reps_target...
✅ Столбцы успешно добавлены.

📊 Контрольный срез данных:


,exercise_id,Exercise_Name,Sex,Age,BodyweightKg,eq_clean,level_idx,goal_clean,Base_Lift,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,predicted_weight,target_reps,weight_target,reps_target
41679,1295,Incline Row DB,M,17.5,109.50,Dumbbell,1,Athletics,Best3DeadliftKg,240.00,170.00,260.00,102.0,12,58.0,12
14720,932,Flat Chest Db Press,M,41.5,92.48,Dumbbell,1,Athletics,Best3DeadliftKg,215.00,145.00,215.00,84.0,12,48.0,12
25137,1933,Prime Leg Extension,M,21.5,72.95,Machine,2,Bodybuilding,Best3SquatKg,210.00,132.50,270.50,42.3,10,50.0,10
36048,3023,Z-Sit Shoulder Press,M,18.5,93.55,Dumbbell,2,Muscle & Sculpting,Best3BenchKg,75.00,100.00,192.50,38.0,15,20.0,14
1339,1609,MMS Barbell Complex,M,26.0,71.95,Barbell,2,Muscle & Sculpting,Best3DeadliftKg,210.00,120.00,262.50,27.6,15,40.0,14
6266,618,Clean And Jerk Complex,F,18.5,66.77,Barbell,2,Bodybuilding,Best3DeadliftKg,57.50,35.00,100.00,50.6,12,60.0,12
21591,2256,Seated Biceps Curl,M,53.0,108.91,Dumbbell,0,Bodybuilding,Best3DeadliftKg,150.00,167.50,180.00,22.0,10,12.0,10
37392,2691,Straight Arm Pull Down,F,33.0,47.90,Machine,2,Muscle & Sculpting,Best3DeadliftKg,77.50,40.00,85.00,10.3,17,15.0,16
31874,1984,Quad extensions,M,23.0,154.80,Machine,2,Powerlifting,Best3SquatKg,190.00,110.00,212.50,51.1,3,55.0,3
33906,484,Cable Preacher Curls,M,19.5,99.39,Machine,0,Muscle & Sculpting,Best3DeadliftKg,205.00,130.00,222.50,31.7,15,40.0,14


In [20]:
import os

# Определяем имя файла
output_filename = "df_gym_pro_final.csv"

# Сохраняем в CSV (используем utf-8-sig для корректного отображения кириллицы в Excel)
df_final_reg.to_csv(output_filename, index=False, encoding='utf-8-sig')

# Проверка размера и пути
file_size = os.path.getsize(output_filename) / (1024 * 1024) # в Мб
print(f"✅ Датасет успешно сохранен!")
print(f"📁 Файл: {os.path.abspath(output_filename)}")
print(f"📊 Размер: {file_size:.2f} MB")
print(f"📑 Строк: {len(df_final_reg)}")

✅ Датасет успешно сохранен!
📁 Файл: /Users/artemmarkov/Projects/training/catboost-regressor/df_gym_pro_final.csv
📊 Размер: 4.85 MB
📑 Строк: 45239
